# 🚴 Zonas de Entrenamiento en Ciclismo — Test de 5 Minutos (PAM)

---

## Fundamento fisiológico

La **Potencia Aeróbica Máxima (PAM)** es la potencia mínima que solicita el VO₂máx. Un esfuerzo máximo de 5 minutos en bicicleta es suficientemente largo para alcanzar el VO₂máx y minimizar el componente anaeróbico.

$$PAM \approx P_{5min} \times 0.95 \qquad FTP \approx P_{5min} \times 0.76$$
$$VO_2max \; (ml/kg/min) \approx \frac{PAM \times 10.8}{Peso} + 7$$

---
### 📋 Instrucciones:
1. Ejecuta la **Celda 1** y la **Celda 2**
2. Ejecuta la **Celda 3**: aparece el formulario interactivo
3. Ajusta los sliders y haz clic en **▶ Calcular PAM y zonas**

> 💡 **Concepto:** Los *widgets* son la interfaz entre el usuario y el código. Cuando mueves un slider, el valor se actualiza en memoria — sin necesidad de editar ninguna variable.

In [ ]:
# ============================================================
# CELDA 1 — Instalación de librerías
# ============================================================
# 📖 Nota de programación:
# Importamos con alias ('as') para abreviar:
#   import ipywidgets as widgets  → usamos 'widgets.Button()' en vez de 'ipywidgets.Button()'
#   import plotly.graph_objects as go → 'go.Bar()' en vez de la ruta completa
# Esto es convención estándar en la comunidad Python.

!pip install ipywidgets plotly openpyxl --quiet

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías listas. Ejecuta la Celda 2.')

In [ ]:
# ============================================================
# CELDA 2 — Funciones y definición de zonas
# ============================================================
# 📖 CONCEPTO: Diccionarios (dict)
# Un diccionario almacena pares clave:valor.
# perfil = {'PAM': 285, 'FTP': 236, 'VO2max': 51.2}
# Se accede con perfil['PAM'] → 285
# Son ideales para agrupar datos relacionados de un objeto.

# 6 zonas basadas en % PAM — modelo Coggan adaptado
ZONAS_DEF = [
    ('Z1 — Recuperación',     30,  55, '#AED6F1', 'Recuperación activa. FC muy baja. Sesiones de <1h.'),
    ('Z2 — Resistencia',      55,  70, '#A9DFBF', 'Base aeróbica. Umbral aeróbico. Sesiones largas.'),
    ('Z3 — Tempo',            70,  82, '#F9E79F', 'Umbral de lactato. Conversación difícil. 20–60 min.'),
    ('Z4 — Sub-PAM / Umbral', 82,  92, '#F0B27A', 'Alto lactato estable. Esfuerzo de competencia.'),
    ('Z5 — PAM / VO₂máx',    92, 105, '#E59866', 'Máximo aeróbico. Intervalos 3–8 min. Muy exigente.'),
    ('Z6 — Anaeróbico',      105, 130, '#C0392B', 'Sobre PAM. Sprints y esfuerzos muy cortos (<2 min).'),
]

def calcular_perfil(P5min, peso):
    """
    Calcula PAM, FTP y VO2max desde la potencia de 5 min.
    Retorna un diccionario con todos los valores.
    """
    PAM    = round(P5min * 0.95, 1)
    FTP    = round(P5min * 0.76, 1)
    vo2max = round((PAM * 10.8 / peso) + 7, 1)
    # Clasificación W/kg PAM (escala Coggan)
    wkg = PAM / peso
    if wkg >= 6.5:   cat = '🏆 Clase mundial'
    elif wkg >= 5.5: cat = '🥇 Élite'
    elif wkg >= 4.5: cat = '🥈 Categoría 1 / Semi-élite'
    elif wkg >= 3.5: cat = '🥉 Amateur avanzado'
    elif wkg >= 2.5: cat = '🚴 Recreativo'
    else:            cat = '🌱 Iniciado'
    return {'PAM': PAM, 'FTP': FTP, 'vo2max': vo2max,
            'PAM_wkg': round(wkg,2), 'FTP_wkg': round(FTP/peso,2),
            'categoria': cat}

def calcular_zonas(PAM):
    """
    Genera las 6 zonas de entrenamiento en Watts a partir de la PAM.
    Usa un bucle 'for' sobre la lista de definición de zonas.
    """
    zonas = []
    for nombre, p_min, p_max, color, desc in ZONAS_DEF:
        w_min = round(PAM * p_min / 100, 1)
        w_max = round(PAM * p_max / 100, 1)
        zonas.append({
            'nombre': nombre, 'p_min': p_min, 'p_max': p_max,
            'w_min': w_min, 'w_max': w_max,
            'color': color, 'desc': desc
        })
    return zonas

print('✅ Funciones definidas:')
print('   calcular_perfil()  → PAM, FTP, VO2max, categoría W/kg')
print('   calcular_zonas()   → 6 zonas en Watts desde PAM')
print('\n➡️  Ejecuta la Celda 3 para ver el formulario interactivo')

In [ ]:
# ============================================================
# CELDA 3 — 🎛️ FORMULARIO INTERACTIVO
# ============================================================
# 📖 CONCEPTO: Output widgets
# widgets.Output() es un área reservada para mostrar resultados.
# Dentro de 'with out_resultado:' podemos mostrar tablas, gráficos, texto.
# clear_output(wait=True) borra el contenido anterior antes de redibujar.
# Esto permite actualizar los resultados cada vez que el usuario calcula.

estilos = {'description_width': '230px'}
ancho   = widgets.Layout(width='520px')

# --- Widgets de entrada ---
w_nombre = widgets.Text(
    value='Ciclista 1',
    description='👤 Nombre:',
    style=estilos, layout=ancho
)
w_peso = widgets.BoundedIntText(
    value=70, min=30, max=150,
    description='⚖️ Peso (kg):',
    style=estilos, layout=ancho
)
w_P5min = widgets.IntSlider(
    value=280, min=80, max=600, step=5,
    description='⚡ Potencia 5 min (W):',
    style=estilos, layout=ancho,
    continuous_update=False
)

# Preview en tiempo real
# 📖 Cada vez que el slider cambia, observe() llama a actualizar_prev()
w_prev = widgets.HTML()
def actualizar_prev(change):
    P5 = w_P5min.value
    p  = w_peso.value
    PAM  = round(P5 * 0.95, 1)
    FTP  = round(P5 * 0.76, 1)
    wkg  = round(PAM / p, 2)
    w_prev.value = (
        f'<div style="padding:6px 0;color:#6c5ce7;font-size:13px">'
        f'⚡ PAM estimada: <b>{PAM} W</b> &nbsp;|&nbsp; '
        f'FTP estimado: <b>{FTP} W</b> &nbsp;|&nbsp; '
        f'W/kg PAM: <b>{wkg}</b></div>'
    )
w_P5min.observe(actualizar_prev, names='value')
w_peso.observe(actualizar_prev, names='value')
actualizar_prev(None)

# Botones
btn_calc = widgets.Button(
    description='▶  Calcular PAM y zonas',
    button_style='success',
    layout=widgets.Layout(width='210px', height='36px')
)
btn_comp = widgets.Button(
    description='📊 Ver comparativa W/kg',
    button_style='warning',
    layout=widgets.Layout(width='210px', height='36px')
)
btn_exp = widgets.Button(
    description='💾 Exportar',
    button_style='info',
    layout=widgets.Layout(width='130px', height='36px'),
    disabled=True
)

out_res   = widgets.Output()
out_graf  = widgets.Output()
out_comp  = widgets.Output()
out_exp   = widgets.Output()
estado    = {}

# ---- Callback principal ----
def on_calcular(b):
    nombre = w_nombre.value.strip() or 'Ciclista'
    peso   = w_peso.value
    P5min  = w_P5min.value

    perfil = calcular_perfil(P5min, peso)
    zonas  = calcular_zonas(perfil['PAM'])
    estado.update({'nombre': nombre, 'peso': peso, 'P5min': P5min,
                   'perfil': perfil, 'zonas': zonas})

    # --- Tabla de resultados HTML ---
    with out_res:
        clear_output(wait=True)
        filas = ''.join([
            f'<tr style="background:{z["color"]}">'  
            f'<td style="padding:7px 10px;font-weight:bold">{z["nombre"]}</td>'
            f'<td style="padding:7px 10px;text-align:center">{z["p_min"]}–{z["p_max"]}%</td>'
            f'<td style="padding:7px 10px;text-align:center">{z["w_min"]} – {z["w_max"]} W</td>'
            f'<td style="padding:7px 10px;text-align:center">{round(z["w_min"]/peso,2)} – {round(z["w_max"]/peso,2)}</td>'
            f'<td style="padding:7px 10px;font-size:11px">{z["desc"]}</td>'
            f'</tr>'
            for z in zonas
        ])
        display(HTML(f"""
        <div style="background:#dfe6e9;border-radius:10px;padding:14px;margin:8px 0;font-family:sans-serif">
          <b>🚴 {nombre}</b> &nbsp;|&nbsp; {peso} kg &nbsp;|&nbsp;
          P5min: <b>{P5min} W</b> &nbsp;|&nbsp;
          PAM: <b>{perfil['PAM']} W ({perfil['PAM_wkg']} W/kg)</b> &nbsp;|&nbsp;
          FTP: <b>{perfil['FTP']} W ({perfil['FTP_wkg']} W/kg)</b> &nbsp;|&nbsp;
          VO₂máx est.: <b>{perfil['vo2max']} ml/kg/min</b><br>
          <span style="color:#6c5ce7">Categoría: <b>{perfil['categoria']}</b></span>
        </div>
        <table style="border-collapse:collapse;width:100%;font-size:13px">
          <tr style="background:#2d3436;color:white">
            <th style="padding:8px">Zona</th><th style="padding:8px">% PAM</th>
            <th style="padding:8px">Watts</th><th style="padding:8px">W/kg</th>
            <th style="padding:8px">Descripción</th>
          </tr>
          {filas}
        </table>
        """))

    # --- Gráfico Plotly ---
    with out_graf:
        clear_output(wait=True)
        fig = make_subplots(rows=1, cols=2,
            subplot_titles=[
                f'Potencia absoluta (W) — PAM: {perfil["PAM"]} W',
                f'Potencia relativa (W/kg) — PAM: {perfil["PAM_wkg"]} W/kg'
            ])

        for z in zonas:
            hover = (f"<b>{z['nombre']}</b><br>"
                     f"{z['p_min']}–{z['p_max']}% PAM<br>"
                     f"{z['w_min']}–{z['w_max']} W<br>"
                     f"{round(z['w_min']/peso,2)}–{round(z['w_max']/peso,2)} W/kg<br>"
                     f"<i>{z['desc']}</i><extra></extra>")

            fig.add_trace(go.Bar(
                x=[z['w_max']-z['w_min']], y=[z['nombre']],
                base=z['w_min'], orientation='h',
                marker_color=z['color'],
                marker_line_color='white', marker_line_width=2,
                hovertemplate=hover, showlegend=False,
                text=f"{z['w_min']}–{z['w_max']}W", textposition='inside'
            ), row=1, col=1)

            w_min_kg = round(z['w_min']/peso, 2)
            w_max_kg = round(z['w_max']/peso, 2)
            fig.add_trace(go.Bar(
                x=[w_max_kg-w_min_kg], y=[z['nombre']],
                base=w_min_kg, orientation='h',
                marker_color=z['color'],
                marker_line_color='white', marker_line_width=2,
                hovertemplate=f"<b>{z['nombre']}</b><br>{w_min_kg}–{w_max_kg} W/kg<extra></extra>",
                showlegend=False,
                text=f"{w_min_kg}–{w_max_kg}", textposition='inside'
            ), row=1, col=2)

        fig.add_vline(x=perfil['PAM'], line_dash='dash', line_color='#2d3436',
                      annotation_text=f"PAM {perfil['PAM']}W",
                      annotation_font_size=10, row=1, col=1)
        fig.add_vline(x=perfil['FTP'], line_dash='dot', line_color='#636e72',
                      annotation_text=f"FTP {perfil['FTP']}W",
                      annotation_font_size=10, row=1, col=1)
        fig.add_vline(x=perfil['PAM_wkg'], line_dash='dash', line_color='#2d3436',
                      annotation_text=f"{perfil['PAM_wkg']} W/kg",
                      annotation_font_size=10, row=1, col=2)

        fig.update_layout(
            title=f'🚴 Zonas de Entrenamiento Ciclismo — {nombre}',
            height=430, plot_bgcolor='#f8f9fa',
            margin=dict(l=185, r=20, t=80, b=40)
        )
        fig.show()

    btn_exp.disabled = False

# ---- Callback comparativa W/kg Coggan ----
def on_comparativa(b):
    with out_comp:
        clear_output(wait=True)
        if not estado: print('⚠️ Primero calcula las zonas'); return
        PAM_wkg = estado['perfil']['PAM_wkg']

        categorias = [
            ('Clase mundial', 6.5, '#C0392B'),
            ('Élite', 5.5, '#E67E22'),
            ('Cat. 1 / Semi-élite', 4.5, '#F1C40F'),
            ('Amateur avanzado', 3.5, '#2ECC71'),
            ('Recreativo', 2.5, '#3498DB'),
            ('Iniciado', 0, '#BDC3C7'),
        ]

        fig = go.Figure()
        for i in range(len(categorias)-1):
            nombre_c, umbral_sup, color = categorias[i]
            umbral_inf = categorias[i+1][1]
            fig.add_trace(go.Bar(
                x=[umbral_sup - umbral_inf], y=['Escala W/kg'],
                base=umbral_inf, orientation='h',
                marker_color=color,
                marker_line_color='white', marker_line_width=2,
                name=nombre_c,
                hovertemplate=f"<b>{nombre_c}</b><br>> {umbral_inf} W/kg<extra></extra>",
                text=nombre_c, textposition='inside'
            ))

        fig.add_vline(x=PAM_wkg, line_dash='dash', line_color='black', line_width=3,
                      annotation_text=f'{estado["nombre"]}\n{PAM_wkg} W/kg',
                      annotation_font_size=12)

        fig.update_layout(
            title=f'Posición en escala W/kg — {estado["nombre"]} ({PAM_wkg} W/kg PAM)',
            barmode='stack', height=200,
            plot_bgcolor='#f8f9fa',
            margin=dict(l=20, r=20, t=60, b=40),
            xaxis_title='W/kg PAM'
        )
        fig.show()

# ---- Callback exportar ----
def on_exportar(b):
    with out_exp:
        clear_output(wait=True)
        if not estado: print('⚠️ Primero calcula'); return
        ts = datetime.now().strftime('%Y%m%d_%H%M')
        fn = f"zonas_ciclismo_{estado['nombre'].replace(' ','_')}_{ts}"
        p  = estado['peso']
        df_zonas = pd.DataFrame([{
            'Zona': z['nombre'], '%PAM': f"{z['p_min']}–{z['p_max']}%",
            'W_min': z['w_min'], 'W_max': z['w_max'],
            'Wkg_min': round(z['w_min']/p,2), 'Wkg_max': round(z['w_max']/p,2)
        } for z in estado['zonas']])
        df_perfil = pd.DataFrame([{
            'Atleta': estado['nombre'], 'Peso_kg': p,
            'P5min_W': estado['P5min'],
            'PAM_W': estado['perfil']['PAM'], 'FTP_W': estado['perfil']['FTP'],
            'PAM_Wkg': estado['perfil']['PAM_wkg'], 'FTP_Wkg': estado['perfil']['FTP_wkg'],
            'VO2max_est': estado['perfil']['vo2max'],
            'Categoria': estado['perfil']['categoria']
        }])
        with pd.ExcelWriter(f'{fn}.xlsx', engine='openpyxl') as w:
            df_perfil.to_excel(w, sheet_name='Perfil', index=False)
            df_zonas.to_excel(w, sheet_name='Zonas', index=False)
        df_zonas.to_csv(f'{fn}.csv', index=False)
        print(f'✅ Exportado: {fn}.xlsx | {fn}.csv')
        print('📥 Descarga desde el panel 📁 izquierdo de Colab')

btn_calc.on_click(on_calcular)
btn_comp.on_click(on_comparativa)
btn_exp.on_click(on_exportar)

# --- Mostrar interfaz ---
display(HTML('<h3 style="color:#2d3436;font-family:sans-serif">📝 Ingresa los datos del atleta</h3>'))
display(w_nombre, w_peso, w_P5min, w_prev)
display(widgets.HBox([btn_calc, btn_comp, btn_exp]))
display(out_res, out_graf, out_comp, out_exp)

---
## 📚 Preguntas de reflexión

1. **Explora el slider:** Cambia la potencia de 200W a 400W manteniendo el mismo peso. ¿Cómo cambia la Z4 en W y en W/kg? ¿Qué cambia si también modificas el peso?

2. **PAM vs FTP:** ¿Por qué FTP ≈ 76% de P5min y no el 100%? ¿Qué significa fisiológicamente esa brecha?

3. **Código y observadores:** En la Celda 3, la función `actualizar_prev` está conectada con `.observe()` tanto al slider de potencia como al de peso. ¿Por qué es necesario conectarla a ambos?

4. **W/kg:** Dos ciclistas tienen PAM de 300W. Uno pesa 60 kg y otro 80 kg. Ingresa ambos en el notebook. ¿Cuál categoría le asigna a cada uno? ¿Por qué W/kg es más relevante que Watts absolutos en ciclismo de montaña?

5. **Limitaciones:** La fórmula de VO₂máx asume eficiencia constante. ¿En qué tipo de ciclistas podría fallar esta estimación?

---
*Notebook — Ciencias del Deporte | github.com/vgarrido1995*